In [11]:
%pip install scikit-learn 

Note: you may need to restart the kernel to use updated packages.


In [12]:
import pandas as pd
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv
from sklearn.preprocessing import StandardScaler
 

In [3]:
load_dotenv()
engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}"
)


Loading Master Dataset

In [6]:
query = """
SELECT 
  f.value,
  t.data_period,
  h.hospital_name, h.state, h.hospital_type,
  m.measure_code, m.measure_name, m.unit,
  rm.reported_measure_name

FROM warehouse.fact_measure_performance f
LEFT JOIN warehouse.dim_hospital h ON f.hospital_id = h.hospital_id
LEFT JOIN warehouse.dim_measure m ON f.measure_id = m.measure_id
LEFT JOIN warehouse.dim_reported_measure rm ON f.reported_measure_id = rm.reported_measure_id
LEFT JOIN warehouse.dim_time t ON f.time_id = t.time_id

  """

df = pd.read_sql(query, engine)


Basic Cleaning

In [7]:
# dropping rows with missing critical missing values
df = df.dropna(subset=['value', 'hospital_name', 'measure_name', 'hospital_type'])
# stripping text fields and standardizing case title
for col in ['hospital_name', 'state', 'hospital_type', 'measure_code', 'measure_name', 'unit', 'reported_measure_name']:
  df[col]= df[col].str.strip().str.title()
# Ensuring value is numeric through coercion
df['value'] = pd.to_numeric(df['value'], errors='coerce')
df = df.dropna(subset=['value'])
# removing duplicates
df = df.drop_duplicates()


Aggregation For Analysis

In [8]:
# Hospital Level Summary
hospital_summary= df.groupby(['hospital_name', 'hospital_type']).agg(
  total_measures= ('value', 'sum'),
  average_measure= ('value', 'mean'),
  measure_count= ('measure_name', 'nunique')
).reset_index()

In [9]:
# Measure Level Summary
measure_summary = df.groupby(['measure_name', 'unit']).agg(
  average_value= ('value', 'mean'),
  total_value= ('value', 'sum'),
  summary_counrt= ('hospital_name', 'nunique')
  
  ).reset_index()

In [ ]:
# Pivot for clustering
hospital_matrix= df.pivot_table(
  index='hospital_name', 
  columns='measure_name', 
  values='value',
  aggfunc='mean'
  # fill missing value with 0
).fillna(0)



Normalization For Clustering

In [13]:
scalar = StandardScaler()
hospital_matrix_scaled = pd.DataFrame(
  scalar.fit_transform(hospital_matrix),
  index=hospital_matrix.index,
  columns=hospital_matrix.columns
)


Validation

In [14]:
print("Dataset shape:", df.shape)
print("Unique hospital types:", df['hospital_type'].unique())
print("Sample hospital summary:")
print(hospital_summary.head())

Dataset shape: (4480288, 9)
Unique hospital types: <StringArray>
['Local Hospital Network']
Length: 1, dtype: str
Sample hospital summary:
                              hospital_name           hospital_type  \
0                                  Adelaide  Local Hospital Network   
1  Adolescent And Young Adult Hospice Manly  Local Hospital Network   
2                           Albany Hospital  Local Hospital Network   
3     Albury Wodonga Health [Albury Campus]  Local Hospital Network   
4    Albury Wodonga Health [Wodonga Campus]  Local Hospital Network   

   total_measures  average_measure  measure_count  
0      1050340.77       171.204689             31  
1      1112191.50       181.938737             31  
2      1314570.30       213.577628             31  
3      1377454.96       223.503969             31  
4      1222628.41       198.801367             31  


Saving Cleaned & Aggregated Data

In [15]:
df.to_csv("../data/processed/cleaned_master_data.csv", index=False)
hospital_summary.to_csv("../data/processed/hospital_summary.csv", index=False)
measure_summary.to_csv("../data/processed/measure_summary.csv", index=False)
hospital_matrix_scaled.to_csv("../data/processed/hospital_matrix_scaled.csv", index=True)

print("All datasets saved!")

All datasets saved!
